In [0]:
dbutils.widgets.text("catalog","dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
df_accounts = spark.read.table(f"{catalog}.bronze.bronze_accounts")

In [0]:
df_branches = spark.read.table(f"{catalog}.silver.silver_branches")
df_customers = spark.read.table(f"{catalog}.silver.customers_scd")

In [0]:
df_cleaned_accounts = df_accounts.join(df_customers.select("customer_id"), on="customer_id", how="left_semi") \
    .join(df_branches.select("branch_id"), on="branch_id", how="left_semi") \
    .withColumn("opening_date", F.to_date(F.col("opening_date"),"yyyy-MM-dd")) \
    .withColumn("closing_date", F.to_date(F.col("closing_date"),"yyyy-MM-dd")) \
        .withColumn("interest_rate", F.col("interest_rate").cast("double")) \
            .withColumn("updated_at", F.to_date(F.col("updated_at"),"yyyy-MM-dd")) \
                .withColumn("account_type", F.lower(F.trim(F.col("account_type")))) \
                    .withColumn("account_status", F.lower(F.trim(F.col("account_status")))) \
                        .withColumn("currency", F.upper(F.trim(F.col("currency")))) \
                            .dropDuplicates() \
                                .dropna(subset=['account_id','customer_id','branch_id','account_type','account_status','opening_date', 'currency','account_tier','interest_rate','updated_at',])

In [0]:
df_cleaned_accounts.write.mode("overwrite").saveAsTable(f"{catalog}.silver.silver_accounts")